# 第1章：从数据记录到任务

**教师参考｜全书深化稿**

## 背景与数据
鸢尾花：一行一朵花，四项尺寸为输入，物种为目标；预先规定的简单阈值不是由数据训练出的算法。

## 本次要回答的问题
比较多数类与给定阈值规则，解释读取数据、训练规则和预测的区别。

先写下预判，再运行，最后用实际结果解释；不需要结果符合预期才算成功。

In [1]:
from pathlib import Path
import os, sys
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "experiments/common.py").exists()
             and (p / "main.tex").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook inside the extracted textbook project.")
sys.path.insert(0, str(root / "experiments"))
print("Project:", root)


Project: /mnt/data/book_v3


## 可检查的完整实验代码

下方与对应`experiments`脚本一致。共享的`common.py`只提供固定数据划分、保存表图和绘图设置；模型、目标、参数及比较规则在此明确写出。阅读函数后，再运行下一单元执行实验。

In [2]:
"""Chapter 1: a complete task with interpretable baseline and an explicit table."""
import numpy as np
import pandas as pd
from common import *

def run():
    Xtr,Xva,ytr,yva,ds=development_trainval('iris')
    # This is a deliberately prescribed rule, not an optimized decision tree.
    positive=(yva==0).to_numpy()
    pred=(Xva['petal length (cm)'].to_numpy()<2.5)
    majority=np.full(len(positive),int(np.mean(ytr==0)>=.5))
    df=table('ch01_task',[{'rule':'majority baseline','n_validation':len(yva),'correct':int(np.sum(majority==positive))},
        {'rule':'prescribed petal threshold','n_validation':len(yva),'correct':int(np.sum(pred==positive))}])
    preview=Xva.iloc[:6].copy();preview['reference_is_setosa']=positive[:6];preview['predicted_is_setosa']=pred[:6]
    table('ch01_records',preview.reset_index(names='original_row'))
    show_result(df)

In [3]:
result = run()
if result is not None:
    print(result)

                      rule  n_validation  correct
         majority baseline            30       20
prescribed petal threshold            30       30


## 检查结果文件与图

下面显示该实验保存的图；所有坐标、样本与数值条件应与正文一致。数据表保存在`results/`。修改代码后，请另存结果或注明变化，不把新配置的输出与正文旧数值混用。

In [4]:
from IPython.display import display, Image
files = sorted((root / "figures").glob("v3_ch01_*.png"))
for path in files:
    print(path.name)
    display(Image(filename=str(path), width=720))
if not files:
    print("This experiment uses tables rather than a new figure.")

This experiment uses tables rather than a new figure.


## 教师解释与核对

30条验证中的简单规则表现好，不等于所有类别问题都容易；本实验只把任务和评价对象讲清。

完整手算解答见`teacher_manual.pdf`。如果结果与这里的参考不同，先检查数据版本、划分、软件环境、目标归一化和随机状态，不要求学生人为调成相同分数。

**评分顺序：**问题与字段角色、数据划分、算法机制、实际结果分析、证据边界。代码能运行但无法说明一项输出的含义，不算完成理解目标。